# Lounge scorecard on served demand, quick first cut

**What changed from v3:**
- **Demand** = *served* women (the Huff share), instead of all women in the 15-min catchment.
- **Cannibalisation** = the share of its served women that siblings take over if it closes, instead of the raw catchment overlap.
- **Capture and rating** are unchanged. Composite ≥ 0.65 is PROTECT, ≤ 0.35 is SHRINK.

The reference is the frozen v3 labels in `data/scenarios/baseline_v3_labels.csv`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_rows", 80)
from src.baseline import run
from src.data_v3 import load_v3
from src.config import REPO_ROOT, load_baseline_assumptions
from src.model import served as S, scorecard, growth
r = run(); v3 = load_v3(); a = load_baseline_assumptions(REPO_ROOT / "data/scenarios/baseline.yaml")
old = pd.read_csv(REPO_ROOT / "data/scenarios/baseline_v3_labels.csv")
old_action = dict(zip(old.id, old.action))

## 1. Signals and anchors

In [2]:
pd.DataFrame([dict(signal=s.name, field=s.field, worst=s.worst, best=s.best, weight=s.weight, why=s.why) for s in scorecard.SIGNALS])

,signal,field,worst,best,weight,why
0,demand,served_women,0.0,20000.00,1.0,Quick first cut (2026-10-11): served women run...
1,cannibalisation,absorbed_share,0.4,0.00,1.0,"Leave-one-out: close the lounge, re-split dema..."
2,capture,capture,0.0,0.15,1.0,Median 6.5% over the scored lounges; 15% is ab...
3,rating,rating_gap,-0.3,0.30,0.5,"Gaps run -0.2 to +0.3, median -0.1: most loung..."


## 2. v3 → new calls

In [3]:
f = {x.branch_id: x for x in r.features}
rows = []
for d in r.decisions:
    if d.action == "NOT SCORED": continue
    x = f[d.branch_id]
    rows.append(dict(lounge=d.branch_id, emirate=x.emirate, v3=old_action.get(d.branch_id), new=d.action, composite=round(d.composite, 3),
                     confidence=d.confidence, served=round(x.served_women or 0), absorbed=round(x.absorbed_share or 0, 2),
                     v3_shared=round(x.shared_share, 2), capture=round(x.capture, 3), **{f"s_{k}": round(v, 2) for k, v in d.scores.items()}))
t = pd.DataFrame(rows).sort_values("composite", ascending=False).set_index("lounge")
print(pd.crosstab(t.v3, t.new)); t

new      HOLD  PROTECT  SHRINK
v3                            
HOLD       11        3       1
PROTECT     0        3       0
SHRINK      1        0       4


,emirate,v3,new,composite,confidence,served,absorbed,v3_shared,capture,s_demand,s_cannibalisation,s_capture,s_rating
lounge,,,,,,,,,,,,,
al-ain,Abu Dhabi,PROTECT,PROTECT,0.873,high,17290,0.00,0.00,0.123,0.86,1.00,0.82,0.75
al-dhafra,Abu Dhabi,HOLD,PROTECT,0.769,low,15202,0.00,0.00,0.709,0.76,1.00,0.85,0.17
ras-al-khaimah,Ras Al Khaimah,PROTECT,PROTECT,0.716,medium,8531,0.00,0.00,0.138,0.43,0.99,0.92,0.33
al-taif-mall,Fujairah,PROTECT,PROTECT,0.714,medium,6213,0.01,0.00,0.143,0.31,0.99,0.95,0.50
khalifa-city-a,Abu Dhabi,HOLD,PROTECT,0.695,low,19353,0.22,0.97,0.126,0.97,0.46,0.84,0.33
al-falah,Abu Dhabi,HOLD,PROTECT,0.683,low,10403,0.17,0.57,0.542,0.52,0.57,0.80,1.00
baniyas,Abu Dhabi,HOLD,HOLD,0.578,low,11458,0.14,0.85,0.094,0.57,0.66,0.62,0.33
shakhbout-city,Abu Dhabi,HOLD,HOLD,0.562,low,13535,0.19,0.98,0.091,0.68,0.52,0.61,0.33
shahama,Abu Dhabi,HOLD,HOLD,0.562,low,12953,0.20,0.82,0.087,0.65,0.49,0.58,0.50


## 3. Where the composites fall against the lines

In [4]:
print(t.composite.describe().round(3))
print("within 0.05 of a line:", list(t.index[(abs(t.composite - scorecard.PROTECT_AT) < 0.05) | (abs(t.composite - scorecard.SHRINK_AT) < 0.05)]))

count    23.000
mean      0.503
std       0.175
min       0.217
25%       0.365
50%       0.468
75%       0.631
max       0.873
Name: composite, dtype: float64
within 0.05 of a line: ['khalifa-city-a', 'al-falah', 'jumeirah-park', 'nad-al-sheba', 'al-barsha', 'city-walk', 'delma', 'mohammed-bin-zayed-city']


## 4. Does capture double-count competition now?

In [5]:
print("Spearman capture vs served:", round(t.capture.corr(t.served, method="spearman"), 2),
      "| capture vs Huff share (served / catchment women):",
      round(t.capture.corr(pd.Series({k: (f[k].served_women or 0) / f[k].addressable_women for k in t.index}), method="spearman"), 2))

Spearman capture vs served: 0.78 | capture vs Huff share (served / catchment women): 0.95


## 5. Open questions for review

1. **Overlap no longer condemns the Abu Dhabi city lounges by itself.** khalifa-city-a and al-falah become PROTECT. Their absorbed share is about 0.2, so siblings would take only a fifth of their customers.
2. **City Walk becomes SHRINK.** It serves very few women because Dubai's premium market is crowded, while v3 gave it full demand credit. Is that what we believe?
3. **Capture double-counts competition.** Section 4 confirms it: capture correlates 0.95 with the Huff share. Drop capture (or move rating into attractiveness), so competition counts once.
4. **The anchors** (20k served, 0.4 absorbed) were set from this run's range: calibrated on the same lounges. Fine for a first look; they need a stated rationale before we adopt them.